In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import GCNConv, GATConv, ChebConv, SAGEConv, global_mean_pool
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_geometric/typing.py:31: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: dlopen(/Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_scatter/_scatter_cpu.so, 0x0006): Symbol not found: __ZN2at4_ops6narrow4callERKNS_6TensorExxx
  Referenced from: <FC4F5CE6-3038-3A9A-B98B-661CD724F01B> /Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch_scatter/_scatter_cpu.so
  Expected in:     <66FB8649-BB87-3CD6-A177-462038DCAE02> /Users/siddharth/opt/anaconda3/lib/python3.9/site-packages/torch/lib/libtorch_cpu.dylib
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "


In [22]:
class BrainConnectivityNet(nn.Module):
    def __init__(self, num_nodes, num_classes):
        super(BrainConnectivityNet, self).__init__()
        # GCN branch
        self.gcn1 = GCNConv(in_channels= num_nodes, out_channels=64)
        self.gcn2 = GCNConv(in_channels=64, out_channels=32)

        # GAT branch
        self.gat1 = GATConv(in_channels= num_nodes, out_channels=64)
        self.gat2 = GATConv(in_channels=64, out_channels=32)

        # Chebyshev branch
        self.cheb1 = ChebConv(in_channels= num_nodes, out_channels=64, K=2)
        self.cheb2 = ChebConv(in_channels=64, out_channels=32, K=2)

        # Pooling layer
        self.pool = global_mean_pool

        # Classifier
        self.classifier = nn.Linear(64*4, num_classes)  # 4 branches with 64 features each

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # GCN branch
        x_gcn = F.relu(self.gcn1(x, edge_index))
        x_gcn = F.relu(self.gcn2(x_gcn, edge_index))

        # GAT branch
        x_gat = F.relu(self.gat1(x, edge_index))
        x_gat = F.relu(self.gat2(x_gat, edge_index))

        # Chebyshev branch
        x_cheb = F.relu(self.cheb1(x, edge_index))
        x_cheb = F.relu(self.cheb2(x_cheb, edge_index))

        # Concatenate the outputs from all branches
        x = torch.cat((x_gcn, x_gat, x_cheb), dim=1)

        # Apply the pooling layer to get graph-level representation
        x = self.pool(x, batch)

        # Apply the classification layer
        x = self.classifier(x)

        return F.log_softmax(x, dim=1)


In [25]:
def get_model_summary(model):
    print("Layer Name (Type) | Input Features | Output Features")
    print("------------------|-----------------|------------------")
    for name, module in model.named_modules():
        if isinstance(module, nn.Module):
            input_feat = module.in_features if hasattr(module, 'in_features') else None
            output_feat = module.out_features if hasattr(module, 'out_features') else None
            print(f"{name} ({type(module).__name__}) | {input_feat} | {output_feat}")


In [27]:
model = BrainConnectivityNet(num_nodes=100, num_classes=10)
edge_index = 32  # Replace with your actual edge_index if needed
batch = 10     # Replace with your actual batch information if needed
features = torch.randn(100, 16)  # Replace 16 with your actual feature dimension
output = model(data=(features, edge_index, batch))
get_model_summary(model)

AttributeError: 'tuple' object has no attribute 'x'

In [9]:
from torchinfo import summary

model = BrainConnectivityNet(num_nodes=100, num_classes=10)
summary(model, input_size=(100, 64))  # Replace 64 with your 

RuntimeError: Failed to run torchinfo. See above stack traces for more details. Executed layers up to: []

In [8]:
%pip install torchinfo

Note: you may need to restart the kernel to use updated packages.
